# Amazon Bedrock Prompt Management

This notebook demonstrates AWS Bedrock's Prompt Management capabilities, which provide a centralized way to create, version, and manage reusable prompt templates. This feature is essential for enterprise applications where consistency, governance, and collaboration are critical.

## Key Benefits of Prompt Management:
- **Reusability**: Create templates once, use across multiple applications
- **Version Control**: Track changes and maintain different versions
- **Collaboration**: Teams can share and iterate on prompts
- **Governance**: Centralized control over prompt content and behavior
- **A/B Testing**: Compare different prompt variants for optimization

## Setup and Client Initialization

We initialize three different Bedrock clients, each serving a specific purpose:
- **bedrock-agent**: Manages prompt templates, versions, and configurations
- **bedrock-runtime**: Executes model inference using managed prompts
- **bedrock-agent-runtime**: Provides advanced features like prompt optimization

In [1]:
import boto3
import json
import datetime
from IPython.display import display, JSON

# Initialize Bedrock clients for different functionalities
bedrock_agent = boto3.client("bedrock-agent")
bedrock_runtime = boto3.client("bedrock-runtime")
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime")

# Using Amazon Nova Micro - a cost-effective model for text generation
model_id = "amazon.nova-micro-v1:0"

## Creating a Prompt Template

Amazon Bedrock makes it easy to create reusable prompt templates through Prompt Management. Instead of writing the same prompt over and over in your code, you can save a template once and use it across different workflows and plug in variables as needed.

When you create a prompt, you choose the model you want to run it with and customize how that model behaves using inference settings. You can even test different versions (called "variants") of your prompt to see which one gives you the best results.

✏️ As you refine your prompt, you can save versions along the way so you can experiment without losing your progress.

When you're ready, you can integrate the prompt into your app by referencing it directly in model inference calls or by adding it to a Bedrock Flow.

### Template Variables
The double curly braces `{{variable_name}}` syntax allows dynamic content injection, making prompts flexible and reusable across different contexts.

### Creating the Prompt Template

This example creates a job description generator template with:
- **Input Variables**: Structured placeholders for dynamic content
- **Inference Configuration**: Controls model behavior (temperature, tokens, etc.)
- **Variants**: Different versions of the same prompt for A/B testing

The template uses a timestamp-based naming convention to ensure uniqueness across multiple runs.

In [2]:
# Generate unique prompt name with timestamp
prompt_name = f"job-description-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

# Define the prompt template with variable placeholders
# Variables use {{variable_name}} syntax for substitution
template_text = '''
You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words
'''

# Create the managed prompt with configuration
response = bedrock_agent.create_prompt(
    name=prompt_name,
    description="Generates inclusive job descriptions from structured inputs",
    defaultVariant="v1",  # Default variant to use
    variants=[
        {
            "name": "v1",
            "modelId": model_id,  # Associate with specific model
            "templateType": "TEXT",
            "templateConfiguration": {
                "text": {
                    # Define all input variables the template expects
                    "inputVariables": [
                        {"name": "job_title"},
                        {"name": "responsibilities"},
                        {"name": "requirements"},
                        {"name": "location"},
                        {"name": "work_type"}
                    ],
                    "text": template_text
                }
            },
            # Configure model inference parameters
            "inferenceConfiguration": {
                "text": {
                    "maxTokens": 500,      # Limit response length
                    "temperature": 0.7,    # Balance creativity vs consistency
                    "topP": 0.9,           # Nucleus sampling parameter
                    "stopSequences": []    # Optional stop sequences
                }
            }
        }
    ]
)

print("\n==================== Response Object ====================\n")
display(JSON(response))

print("\n==================== Prompt ARN ====================\n")
print(response['arn'])

## Prompt Versioning Strategy

When you save a prompt, it starts out as a draft. You can keep updating and tweaking that draft. Whether it's the wording, variables, or configuration, you can modify and update the prompt until you're happy with it.

Once you're ready to use the prompt in a real application, you can create a version of it.

✏️ A version captures exactly how your prompt looked at that moment. You should create a version when you're confident in the prompt and want to lock it in for production use.

Having versions makes it easy to manage changes. You can switch between different prompt versions, test variations, and update your app with the one that fits your use case best.

### Best Practices for Versioning:
- Create versions for stable, tested prompts
- Use descriptive version descriptions
- Keep drafts for experimental changes
- Version numbers are automatically incremented

Create a new prompt version by running the following code.

In [3]:
# Create a version from the current draft
# Versions are immutable snapshots of the prompt configuration
response = bedrock_agent.create_prompt_version(
    description='Initial prompt for creating job description documents.',
    promptIdentifier=response['arn']  # Use the ARN from previous response
)

print("\n==================== PROMPT VERSION ARN ====================\n")
print(response["arn"])

## Using Managed Prompts in Inference

To invoke the model using a prompt created using Prompt Management you need to use the converse API as documented in the boto3 documentation, not the direct invoke_model API.

### Key Differences:
- **Converse API**: Required for managed prompts, provides unified interface
- **Prompt Variables**: Passed as a dictionary with structured format
- **Model ID**: Use the prompt ARN with version number

The converse API automatically handles the prompt template substitution and applies the configured inference settings.

In [5]:
# Use the versioned prompt ARN for inference
# This ensures consistent behavior in production
prompt_arn_with_version = response["arn"]  # From the version creation above

# Execute inference using the managed prompt
response = bedrock_runtime.converse(
    modelId=prompt_arn_with_version,  # Use prompt ARN as model ID
    promptVariables={  # Provide values for template variables
        "job_title": {"text": "UX Designer"},
        "responsibilities": {"text": "Design user interfaces, run usability testing, collaborate with product teams"},
        "requirements": {"text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"},
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"}
    }
)

print("\n==================== Response Text ====================\n")
print(response['output']['message']['content'][0]['text'])

## Prompt Optimization

Optimization rewrites prompts to yield inference results that are more suitable for your use case. You can choose the model that you want to optimize the prompt for and then generate a revised prompt.

After you submit a prompt to optimize, Amazon Bedrock analyzes the components of the prompt. If the analysis is successful, it then rewrites the prompt. You can then copy and use the text of the optimized prompt and create a new variant for the prompt.

### How Optimization Works:
1. **Analysis**: Bedrock examines prompt structure, clarity, and effectiveness
2. **Rewriting**: AI-powered optimization improves prompt quality
3. **Streaming Response**: Optimized prompt is returned via event stream
4. **Integration**: Use optimized version to create new prompt variants

This feature helps improve prompt effectiveness without manual trial-and-error.

In [6]:
# Function to handle streaming optimization response
def handle_response_stream(response):
    try:
        event_stream = response['optimizedPrompt']
        for event in event_stream:
            if 'optimizedPromptEvent' in event:
                print("\n==================== OPTIMIZED PROMPT ====================\n")
                print(event['optimizedPromptEvent']['optimizedPrompt']['textPrompt']['text'])
    except Exception as e:
        raise e

# Prepare prompt for optimization
prompt_input = {
    "textPrompt": {
        "text": template_text
    }
}

# Submit prompt for AI-powered optimization
response = bedrock_agent_runtime.optimize_prompt(
            input=prompt_input,
            targetModelId=model_id  # Optimize for specific model
        )

print("\n==================== ORIGINAL PROMPT ====================\n")
print(template_text)
handle_response_stream(response)

## Resource Cleanup

It's important to clean up resources after experimentation to avoid unnecessary costs. This section demonstrates how to list and delete managed prompts programmatically.

In [12]:
# Clean up: Delete all prompts created during this session
# This is important for cost management and resource cleanup
response = bedrock_agent.list_prompts()

for prompt in response['promptSummaries']:
    prompt_id = prompt['id']  # The correct key is 'id', not 'promptId'
    print(f"Deleting prompt: {prompt['name']} (ID: {prompt_id})")
    bedrock_agent.delete_prompt(promptIdentifier=prompt_id)

print("All prompts deleted successfully")

## Summary and Best Practices

This notebook demonstrated the complete lifecycle of prompt management in Amazon Bedrock:

### Key Takeaways:
1. **Centralized Management**: Prompts are stored and versioned centrally
2. **Template Variables**: Enable dynamic content injection
3. **Version Control**: Maintain history and enable rollbacks
4. **Optimization**: AI-powered prompt improvement
5. **Inference Integration**: Seamless use with converse API

### Production Best Practices:
- **Version Management**: Always use versioned prompts in production
- **Testing**: Compare prompt versions with consistent test data
- **Documentation**: Maintain clear descriptions for each version
- **Governance**: Implement approval processes for prompt changes
- **Monitoring**: Track prompt performance and usage metrics
- **Cleanup**: Regularly remove unused prompts to manage costs

### Integration Patterns:
- **CI/CD**: Automate prompt deployment with infrastructure as code
- **A/B Testing**: Use different versions for performance comparison
- **Multi-Environment**: Maintain separate prompts for dev/staging/prod
- **Team Collaboration**: Share and iterate on prompts across teams